# Step 1: Download Bible Data and Create ASR Manifest

This notebook guides you through downloading audio and text data for Runyoro (nyo) and Rutooro (ttj) from the Digital Bible Platform (DBP) / Bible Brain, and then preparing an ASR (Automatic Speech Recognition) manifest file suitable for use with ASR toolkits like SpeechBrain.

## Setup and Configuration

1.  **API Key**: You need a DBP API key. You can obtain one by registering at [https://www.faithcomesbyhearing.com/audio-bibles/bible-brain](https://www.faithcomesbyhearing.com/audio-bibles/bible-brain) (or their developer portal if the link changed).
2.  **Python Environment**: Ensure you have Python installed. It's recommended to use a virtual environment.
3.  **Required Libraries**: The scripts used in this notebook require certain Python libraries. Install them by running:
    ```bash
    pip install requests tqdm soundfile PyYAML notebook ipywidgets
    ```
    You might also need `libsndfile` for `soundfile` to work correctly:
    ```bash
    # On Debian/Ubuntu
    sudo apt-get update
    sudo apt-get install libsndfile1
    
    # On macOS (using Homebrew)
    # brew install libsndfile
    ```
4.  **Project Structure**: This notebook assumes the following project structure:
    ```
    runyoro-rutooro-tts/
    ├── notebooks/
    │   └── 1_data_download_and_manifest_creation.ipynb
    ├── runyoro_speech_ai/
    │   ├── data_ingestion/
    │   │   ├── __init__.py
    │   │   └── download_bible_brain.py
    │   ├── asr_finetune/
    │   │   ├── __init__.py
    │   │   └── build_manifest.py
    │   └── __init__.py
    └── data/  (This will be created by the scripts)
    ```

In [ ]:
# Import necessary libraries for the notebook
import os
from pathlib import Path
import subprocess
import sys
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

# Define project base directory (assuming notebook is in 'notebooks' folder)
PROJECT_ROOT = Path.cwd().parent # This should point to 'runyoro-rutooro-tts'
DATA_INGESTION_DIR = PROJECT_ROOT / "runyoro_speech_ai" / "data_ingestion"
ASR_FINETUNE_DIR = PROJECT_ROOT / "runyoro_speech_ai" / "asr_finetune"
DEFAULT_DATA_DIR = PROJECT_ROOT / "data" / "bible"

# Add the project's Python modules to the path for direct import if needed
sys.path.append(str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Ingestion Scripts Dir: {DATA_INGESTION_DIR}")
print(f"ASR Finetune Scripts Dir: {ASR_FINETUNE_DIR}")
print(f"Default Data Dir: {DEFAULT_DATA_DIR}")

# Ensure the scripts exist
downloader_script = DATA_INGESTION_DIR / "download_bible_brain.py"
manifest_script = ASR_FINETUNE_DIR / "build_manifest.py"

if not downloader_script.exists():
    display(Markdown(f"<font color='red'>**Error:** `download_bible_brain.py` not found at `{downloader_script}`. Please ensure the project structure is correct.</font>"))
if not manifest_script.exists():
    display(Markdown(f"<font color='red'>**Error:** `build_manifest.py` not found at `{manifest_script}`. Please ensure the project structure is correct.</font>"))

DEFAULT_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Configuration Parameters

Please fill in your DBP API key and review the other parameters.

In [ ]:
# --- User Configuration ---
api_key_input = widgets.Password(
    value=os.environ.get('DBP_API_KEY', ''),
    placeholder='Enter your DBP API Key',
    description='DBP API Key:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

language_codes_input = widgets.Text(
    value='nyo,ttj',
    placeholder='e.g., nyo,ttj',
    description='Language Codes:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

 # Specific filesets (Optional - if you know them, otherwise leave blank to discover)
audio_fileset_ids_input = widgets.Text(
    value='NYOBSNN2DA,TTJBSBN2DA', # Example Runyoro Audio & Rutooro Audio (Non-Drama)
    placeholder='e.g., NYOBSNN2DA,TTJBSBN2DA (comma-separated)',
    description='Audio Fileset IDs (Optional):',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px')
)

text_fileset_id_runyoro_input = widgets.Text(
    value='NYOTBTN2ET', # Example Runyoro Text (Nyoro Bible 1912 Edition (OT Portion) New Testament in Runyoro [NYOTBTN])
    placeholder='e.g., NYOTBTN2ET for Runyoro',
    description='Runyoro Text Fileset ID:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

text_fileset_id_rutooro_input = widgets.Text(
    value='TTJOSTN2ET', # Example Rutooro Text (Tooro Bible 1900 Edition New Testament in Tooro [TTJOSTN])
    placeholder='e.g., TTJOSTN2ET for Rutooro',
    description='Rutooro Text Fileset ID:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

books_to_download_input = widgets.Text(
    value='MAT,MRK,LUK,JHN', # Gospels are a good start
    placeholder='e.g., MAT,MRK,LUK,JHN (comma-separated, blank for all)',
    description='Books to Download (Optional):',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px')
)

data_dir_input = widgets.Text(
    value=str(DEFAULT_DATA_DIR),
    placeholder='Path to data directory',
    description='Data Directory:',
    disabled=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

display(Markdown("### Data Download Parameters"))
display(api_key_input)
display(language_codes_input)
display(audio_fileset_ids_input)
display(text_fileset_id_runyoro_input)
display(text_fileset_id_rutooro_input)
display(books_to_download_input)
display(data_dir_input)

display(Markdown("### Manifest Generation Parameters"))
manifest_filename_nyo_input = widgets.Text(
    value="runyoro_speechbrain_manifest_gospels.json", 
    description='Runyoro Manifest Filename:', 
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
manifest_filename_ttj_input = widgets.Text(
    value="rutooro_speechbrain_manifest_gospels.json", 
    description='Rutooro Manifest Filename:', 
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
display(manifest_filename_nyo_input)
display(manifest_filename_ttj_input)

## Helper Function to Run Commands

In [ ]:
def run_command(command, cwd=None):
    """Runs a shell command and prints its output in real-time."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True, bufsize=1, universal_newlines=True, cwd=cwd)
    print(f"Running command: {command}\nIn directory: {cwd or os.getcwd()}\n")
    output = []
    for line in iter(process.stdout.readline, ''):
        print(line, end='') # Print in real-time
        output.append(line)
    process.stdout.close()
    return_code = process.wait()
    if return_code:
        display(Markdown(f"<font color='red'>**Command failed with error code {return_code}**</font>"))
    else:
        display(Markdown(f"<font color='green'>**Command completed successfully.**</font>"))
    return return_code, "".join(output)

## Part 1: Download Data from Bible Brain

This section runs the `download_bible_brain.py` script using the parameters you provided. 
It will attempt to download audio for the specified languages/filesets and corresponding text data.

In [ ]:
def execute_download():
    api_key = api_key_input.value
    dest_dir = Path(data_dir_input.value)
    langs = language_codes_input.value
    audio_fsets = audio_fileset_ids_input.value
    text_fset_nyo = text_fileset_id_runyoro_input.value
    text_fset_ttj = text_fileset_id_rutooro_input.value
    books = books_to_download_input.value

    if not api_key:
        display(Markdown("<font color='red'>**Error:** API Key is required.</font>"))
        return

    final_dl_cmd_list = [
        sys.executable, str(downloader_script),
        "--api_key", api_key,
        "--dest", str(dest_dir),
        "--language_codes", langs,
    ]
    if audio_fsets: final_dl_cmd_list.extend(["--fileset_ids_audio", audio_fsets])
    if books: final_dl_cmd_list.extend(["--book_ids", books])

    # The download_bible_brain.py script is designed to download all specified audio.
    # For text, it uses --fileset_id_text if provided. If not, it tries to discover text for all --language_codes.
    # If you provide --fileset_id_text, it will be used for text downloads related to the languages it's relevant for.
    # For this notebook, we'll use the Runyoro text fileset ID if available, otherwise the Rutooro one, or let it discover.
    # This might mean that if you download audio for both NYO and TTJ, and provide only NYO's text_id,
    # TTJ text might not be downloaded using its specific ID unless discovery finds it or it matches NYO's.
    # For precise control over multiple languages' text downloads with specific IDs, separate calls to the download script might be best.

    if text_fset_nyo:
        final_dl_cmd_list.extend(["--fileset_id_text", text_fset_nyo])
        display(Markdown(f"Attempting text download using Runyoro text fileset ID: {text_fset_nyo} (and discovering for other specified languages if this ID is not relevant to them)."))
    elif text_fset_ttj:
        final_dl_cmd_list.extend(["--fileset_id_text", text_fset_ttj])
        display(Markdown(f"Runyoro text fileset ID not specified. Attempting text download using Rutooro text fileset ID: {text_fset_ttj} (and discovering for other specified languages if this ID is not relevant to them)."))
    else:
        display(Markdown(f"No specific text fileset ID provided. The script will attempt to discover text filesets for all languages: {langs}"))
        
    display(Markdown(f"**Download Command:** `{' '.join(final_dl_cmd_list)}`"))
    display(Markdown("This may take a significant amount of time depending on the number of books and filesets."))
    run_command(" ".join(final_dl_cmd_list), cwd=PROJECT_ROOT)
    
    display(Markdown("**Note on Text Downloads for Multiple Languages:** The download script (`download_bible_brain.py`) currently uses a single `--fileset_id_text` for targeted text download. "
                     "If you need to use distinct text fileset IDs for different languages (e.g., one for Runyoro and another for Rutooro), you might need to run the download process multiple times with focused parameters. "
                     "For example, after the main download, to ensure Rutooro text is fetched using its specific ID, you could run: <br>"
                     f"`{sys.executable} {downloader_script} --api_key YOUR_KEY --dest {dest_dir} --language_codes ttj --skip_audio_download --fileset_id_text {text_fset_ttj or 'RUTOORO_TEXT_ID'} --book_ids '{books or ''}'`"))

download_button = widgets.Button(description="Start Download")
download_output = widgets.Output()

def on_download_button_clicked(b):
    with download_output:
        download_output.clear_output()
        display(Markdown("Starting download process..."))
        execute_download()

download_button.on_click(on_download_button_clicked)
display(download_button, download_output)

## Part 2: Build ASR Manifest File

This section runs the `build_manifest.py` script (located in `runyoro_speech_ai/asr_finetune/`). It requires the audio and text data to have been downloaded successfully in Part 1 and creates a manifest in **SpeechBrain JSON format**.

You need to specify which audio fileset and which text fileset to use for creating each manifest. This is important if you downloaded data for multiple languages or versions.

In [ ]:
# --- Manifest Configuration ---
display(Markdown("### Configure Paths for SpeechBrain Manifest Generation"))
display(Markdown("Verify that the `Audio Fileset ID for Manifest` and `Text Fileset ID for Manifest` below match the directory names of your downloaded data. These are often the same IDs used for download."))

# Runyoro Manifest
audio_fileset_id_nyo_manifest_input = widgets.Text(
    value='NYOBSNN2DA', # Should match a directory in data_dir / 'audio' / NYOBSNN2DA
    description='Runyoro Audio Fileset ID (for manifest):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
text_fileset_id_nyo_manifest_input = widgets.Text(
    value=text_fileset_id_runyoro_input.value, # Default to the one used for download, e.g., data_dir / 'text' / NYOTBTN2ET
    description='Runyoro Text Fileset ID (for manifest):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

# Rutooro Manifest
audio_fileset_id_ttj_manifest_input = widgets.Text(
    value='TTJBSBN2DA', # Should match a directory in data_dir / 'audio' / TTJBSBN2DA
    description='Rutooro Audio Fileset ID (for manifest):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
text_fileset_id_ttj_manifest_input = widgets.Text(
    value=text_fileset_id_rutooro_input.value, # Default to the one used for download
    description='Rutooro Text Fileset ID (for manifest):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

display(Markdown("#### Runyoro Manifest Details"))
display(audio_fileset_id_nyo_manifest_input)
display(text_fileset_id_nyo_manifest_input)

display(Markdown("#### Rutooro Manifest Details"))
display(audio_fileset_id_ttj_manifest_input)
display(text_fileset_id_ttj_manifest_input)

def execute_manifest_creation(lang_code_short):
    data_dir = Path(data_dir_input.value)
    audio_base = data_dir / "audio"
    text_base = data_dir / "text"

    if lang_code_short == "nyo":
        audio_fset_id = audio_fileset_id_nyo_manifest_input.value
        text_fset_id = text_fileset_id_nyo_manifest_input.value
        manifest_file = data_dir / manifest_filename_nyo_input.value # Value from earlier cell
        actual_lang_code_for_script = "nyo"
    elif lang_code_short == "ttj":
        audio_fset_id = audio_fileset_id_ttj_manifest_input.value
        text_fset_id = text_fileset_id_ttj_manifest_input.value
        manifest_file = data_dir / manifest_filename_ttj_input.value # Value from earlier cell
        actual_lang_code_for_script = "ttj"
    else:
        display(Markdown(f"<font color='red'>**Invalid language code for manifest: {lang_code_short}**</font>"))
        return

    if not audio_fset_id or not text_fset_id:
        display(Markdown(f"<font color='red'>**Error:** Audio and Text fileset IDs for {actual_lang_code_for_script.upper()} manifest generation must be specified.</font>"))
        return
    
    # Check if the specific audio and text directories exist before running the script
    path_to_audio_data = audio_base / audio_fset_id
    path_to_text_data = text_base / text_fset_id

    if not path_to_audio_data.exists():
        display(Markdown(f"<font color='red'>**Error:** Audio directory `{path_to_audio_data}` not found for {actual_lang_code_for_script.upper()}. Ensure data was downloaded and Fileset ID is correct.</font>"))
        return
    if not path_to_text_data.exists():
        display(Markdown(f"<font color='red'>**Error:** Text directory `{path_to_text_data}` not found for {actual_lang_code_for_script.upper()}. Ensure data was downloaded and Fileset ID is correct.</font>"))
        return

    cmd = [
        sys.executable, str(manifest_script), # Points to runyoro_speech_ai/asr_finetune/build_manifest.py
        "--audio_dir_base", str(audio_base),
        "--text_dir_base", str(text_base),
        "--audio_fileset_id", audio_fset_id,
        "--text_fileset_id", text_fset_id,
        "--manifest_path", str(manifest_file),
        "--language_code", actual_lang_code_for_script
    ]

    display(Markdown(f"### Creating {actual_lang_code_for_script.upper()} SpeechBrain ASR Manifest"))
    display(Markdown(f"**Command:** `{' '.join(cmd)}`"))
    run_command(" ".join(cmd), cwd=PROJECT_ROOT)

manifest_button_nyo = widgets.Button(description="Create Runyoro SpeechBrain Manifest")
manifest_output_nyo = widgets.Output()
manifest_button_ttj = widgets.Button(description="Create Rutooro SpeechBrain Manifest")
manifest_output_ttj = widgets.Output()

def on_manifest_button_nyo_clicked(b):
    with manifest_output_nyo:
        manifest_output_nyo.clear_output()
        execute_manifest_creation("nyo")

def on_manifest_button_ttj_clicked(b):
    with manifest_output_ttj:
        manifest_output_ttj.clear_output()
        execute_manifest_creation("ttj")

manifest_button_nyo.on_click(on_manifest_button_nyo_clicked)
manifest_button_ttj.on_click(on_manifest_button_ttj_clicked)

display(Markdown("Click the buttons below to generate the SpeechBrain manifest for each language. "
                 "Ensure the paths and fileset IDs above are correctly set to match your downloaded data."))
display(manifest_button_nyo, manifest_output_nyo)
display(manifest_button_ttj, manifest_output_ttj)

## Next Steps

Once the manifest file(s) (`.json`) are created, you can use them to train an ASR model with toolkits like SpeechBrain.

The SpeechBrain manifest file is a single JSON object where keys are utterance IDs, like this:
```json
{
  "NYOBSNN2DA_MAT_001": {
    "wav": "/abs/path/to/project/data/bible/audio/NYOBSNN2DA/NYOBSNN2DA_MAT_001.mp3",
    "duration": 180.5,
    "words": "transcribed text of Matthew chapter 1..."
  },
  "NYOBSNN2DA_MAT_002": {
    "wav": "/abs/path/to/project/data/bible/audio/NYOBSNN2DA/NYOBSNN2DA_MAT_002.mp3",
    "duration": 150.2,
    "words": "transcribed text of Matthew chapter 2..."
  }
}
```

This file will be used as input for SpeechBrain's data loaders.

**Important Considerations:**
-   **Text Normalization**: The quality of the transcripts is crucial. The current `build_manifest.py` script does very basic normalization. You might need to enhance it to remove verse numbers, special characters, or convert numbers to words, depending on your ASR model's requirements and the source text's characteristics.
-   **Data Splitting**: For training, you'll typically need to split your manifest into training, validation, and possibly test sets. SpeechBrain recipes or custom scripts might be needed for this, often by preparing separate JSON files for train, dev, and test sets.
-   **Audio Quality**: Ensure the downloaded audio is of reasonable quality for ASR. Background noise, music, or dramatization can affect model performance.
-   **Resource Requirements**: Training ASR models, especially for new languages, requires significant computational resources (GPUs) and time.